# Pod 10: sort them

Nobody has told the computer what a "hot Jupiter" is. Today you give it 2,500 planets, two numbers per planet, and ask it to sort them into groups by itself. Then you find out whether the groups it made mean anything.

The method is called **k-means**, and it uses only things you already know: distances, masks, `.mean()`, a loop, and a loss that goes down, as in Pod 9. Sorting data into groups without being told the answer is one of the basic jobs of machine learning.

Run the first cell; it must print a path containing `astr119`. Before every code cell: write your prediction on the sheet, run, compare.

**How to work the "your code here" cells:** one sentence of what you expect in a text cell, then type it or ask Claude for that cell only, read it, run, compare.

In [ ]:
import sys
print(sys.executable)   # must contain 'astr119'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Part 1: look first

The catalog from Pod 8. Masses are in Jupiter masses (the Earth is 0.003), periods in days.

In [ ]:
# 1.1  Keep the planets that have both a mass and a period. Predict how many.
cat = pd.read_csv("../module-4/data/exoplanet_eu_catalog_2024-02.csv", low_memory=False)   # the catalog from Pod 8, one folder over
has_mass = cat["mass"] > 0                      # False for empty cells too
has_period = cat["orbital_period"] > 0
keep = has_mass & has_period
planets = cat[keep]
print(len(planets))

x = np.log10(planets["orbital_period"].values)   # log of the period, days
y = np.log10(planets["mass"].values)             # log of the mass, Jupiter masses

In [ ]:
# 1.2  Look before the computer does. How many groups do YOU see, and where is the middle of each?
fig, ax = plt.subplots()
ax.scatter(x, y, s=3, color="gray")
ax.set_xlabel("log period (days)")
ax.set_ylabel("log mass (Jupiter masses)")
ax.grid(True)                                    # grid lines help you read off positions
fig.savefig("planets_look.png")

## Part 2: let the computer sort them

k-means, in four lines:

1. Put down three **centers** anywhere.
2. Every planet joins the group of the center nearest to it.
3. Every center moves to the middle (the mean) of its group.
4. Repeat 2 and 3 until nothing changes.

The **loss** is the sum of the squared distances from each planet to its own center. A small loss means tight groups. Each round should make it smaller, the way each step of gradient descent did.

Run the next cell once: it only defines three functions.

In [ ]:
# 2.0  The three pieces. Read them; nothing happens until you call them.
def sort_step(x, y, cx, cy):
    # squared distance from every planet to each center
    d0 = (x - cx[0])**2 + (y - cy[0])**2
    d1 = (x - cx[1])**2 + (y - cy[1])**2
    d2 = (x - cx[2])**2 + (y - cy[2])**2
    # a mask for each group: True where that center is the nearest
    in0 = (d0 <= d1) & (d0 <= d2)
    in1 = (d1 < d0) & (d1 <= d2)
    in2 = (d2 < d0) & (d2 < d1)
    loss = np.sum(d0[in0]) + np.sum(d1[in1]) + np.sum(d2[in2])
    return in0, in1, in2, loss               # a function can hand back several things at once

def move_step(x, y, in0, in1, in2):
    # each center moves to the mean of its own group
    cx = np.array([x[in0].mean(), x[in1].mean(), x[in2].mean()])
    cy = np.array([y[in0].mean(), y[in1].mean(), y[in2].mean()])
    return cx, cy

def show(x, y, in0, in1, in2, cx, cy, filename):
    fig, ax = plt.subplots()
    ax.scatter(x[in0], y[in0], s=3, label="group 0")
    ax.scatter(x[in1], y[in1], s=3, label="group 1")
    ax.scatter(x[in2], y[in2], s=3, label="group 2")
    ax.scatter(cx, cy, s=150, color="black", marker="x", label="centers")
    ax.set_xlabel("log period (days)")
    ax.set_ylabel("log mass (Jupiter masses)")
    ax.legend()
    fig.savefig(filename)

In [ ]:
# 2.1  One round by hand. Put YOUR three centers where you saw groups in 1.2 (log period, log mass).
cx = np.array([0.0, 1.0, 2.0])     # YOUR three x positions
cy = np.array([0.0, 0.0, 0.0])     # YOUR three y positions

in0, in1, in2, loss = sort_step(x, y, cx, cy)
print("loss before moving:", round(loss, 1))
show(x, y, in0, in1, in2, cx, cy, "sort_round1.png")

cx, cy = move_step(x, y, in0, in1, in2)
print("centers moved to x:", cx.round(2))
print("                 y:", cy.round(2))

In [ ]:
# 2.2  Now let it run. Predict: after how many rounds does the loss stop falling?
history = []
for step in range(10):
    in0, in1, in2, loss = sort_step(x, y, cx, cy)
    history.append(loss)
    print(step, round(loss, 1))
    cx, cy = move_step(x, y, in0, in1, in2)

show(x, y, in0, in1, in2, cx, cy, "sort_final.png")

fig, ax = plt.subplots()
ax.plot(history, "o-")
ax.set_xlabel("round")
ax.set_ylabel("loss")
fig.savefig("sort_loss.png")

## Part 3: what did it find?

The computer only knows that some planets sit near each other in this plot. Do its groups mean anything? For each group: how many planets, a typical period and mass, and how they were found.

In [ ]:
# 3.1  Describe each group. Before running: which group will hold the most Earth-like masses?
groups = [in0, in1, in2]
for i in range(3):
    g = planets[groups[i]]
    print("group", i, ":", len(g), "planets")
    print("   typical period (days):", round(g["orbital_period"].median(), 1))
    print("   typical mass (Jupiter masses):", round(g["mass"].median(), 3))
    print("   typical mass (Earth masses):", round(g["mass"].median() * 318, 1))   # 1 Jupiter = 318 Earths
    print(g["detection_type"].value_counts().head(2))                            # the two most common ways they were found
    print()

**Name the three groups** on the sheet, in your own words. A period of a few days means the planet is very close to its star, much closer than Mercury (88 days). The Earth is 1 Earth mass; Neptune is 17.

Compare with Pod 8: there, you colored planets by how they were found. Here, nobody told the computer anything about that.

## Part 4: make it yours

Pick one as a pod, or combine, or invent something; each of you in your own notebook. Predict before each run.

- **A bad start.** Put all three starting centers in one corner, for example `cx = np.array([5.0, 6.0, 7.0])` and `cy = np.array([2.0, 2.0, 2.0])`, and run Part 2 again. Same groups? Same final loss? Remember where gradient descent started in Pod 9.
- **The odd ones.** Which planets are farthest from their own center? Print the six farthest with their names, periods, masses and how they were found, then look one up. Is it a strange planet or a strange catalog row?
- **More groups.** Ask for 2, 4 or 5 groups with `sort_step_k` below. Does a new group mean something, or does the computer split a group just because you asked?
- **Forget the logs.** Sort the raw periods and masses instead of their logs. What happens to the groups, and why?
- **Radius instead of mass.** Use the `radius` column (Jupiter radii) in place of `mass`. Are the groups as clear?
- **A combination, or something else** that sorts a data set into groups. Write what you did on the sheet.

A few pods show their notebook at the end.

In [ ]:
# 4.0  For "more groups": the same two steps for any number of centers. Run once, then use it in your own cell.
def sort_step_k(x, y, cx, cy):
    k = len(cx)
    distances = []
    for i in range(k):
        distances.append((x - cx[i])**2 + (y - cy[i])**2)
    distances = np.array(distances)            # one row per center, one column per planet
    nearest = np.argmin(distances, axis=0)     # new trick: for each planet (column), the row number of the smallest distance
    loss = np.sum(np.min(distances, axis=0))   # each planet's distance to its own center, added up
    return nearest, loss

def move_step_k(x, y, nearest, k):
    cx = []
    cy = []
    for i in range(k):
        mine = nearest == i                    # a mask: the planets in group i
        cx.append(x[mine].mean())
        cy.append(y[mine].mean())
    return np.array(cx), np.array(cy)

# Example: five groups, ten rounds. Change the starting centers and the number of rounds.
cx5 = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
cy5 = np.array([0.0, -1.0, 0.0, 1.0, 1.0])
for step in range(10):
    nearest, loss5 = sort_step_k(x, y, cx5, cy5)
    cx5, cy5 = move_step_k(x, y, nearest, len(cx5))
print("final loss with five groups:", round(loss5, 1))

In [ ]:
# 4.1  Your code here. (A text cell above it first: which option, and what you expect.)

## Wrap up

Restart the kernel and Run All; check it runs top to bottom. Fill in the sheet. Then upload: **your `sorting.ipynb`** (with its outputs) and the sheet photo.